<a href="https://colab.research.google.com/github/Supimraid/DSA_Internal_Grp_4/blob/main/Models/catboostrag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install langchain-classic openai langchain-openai pandas scikit-learn catboost xgboost seaborn matplotlib joblib sentence-transformers faiss-cpu torch langchain-community tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 111.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.metrics.pairwise import cosine_similarity
from catboost import CatBoostRegressor, Pool
from sentence_transformers import SentenceTransformer
from pydantic import BaseModel, Field
from langchain_core.tools import Tool  # Changed import path
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_classic.agents.agent import AgentExecutor
from langchain.agents import create_agent
from langchain_classic import hub
from langchain_core.documents import Document
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
import matplotlib.pyplot as plt
import pickle
import faiss
import torch
from openai import OpenAI
from tqdm import tqdm
import warnings
from dotenv import load_dotenv
import os
import time
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import KFold # For defining cross-validation folds
from joblib import dump # For saving the final tuned model
RANDOM_STATE = 42
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.utils._auth")

In [ ]:
# Load environment variables from .env file (if uploaded to Colab session)
load_dotenv()
print("Environment variables from .env file loaded.")

try:
    # Initialize OpenAI Client (will use OPENAI_API_KEY)
    client = OpenAI()
    print("OpenAI Client initialized successfully.")
except Exception as e:
    print("ERROR: Failed to initialize OpenAI client. Ensure 'OPENAI_API_KEY' environment variable is set.")
    raise e

Environment variables from .env file loaded.
OpenAI Client initialized successfully.


In [ ]:
df = pd.read_csv('car_prices_extended_eda.csv')
# --- 1. Define the High-Accuracy 15 Input Features ---
## check if interior and color affects MAE
FINAL_INPUT_FEATURES = [
    'make', 'model', 'body', 'transmission', 'state',
    'condition_category', 'odometer', 'mmr',
    'car_age', 'sale_year', 'sale_month', 'seller_category'
]
TARGET_COLUMN = 'sellingprice'

ALL_COLS_TO_KEEP = FINAL_INPUT_FEATURES + [TARGET_COLUMN]

In [ ]:
# --- 1. Filter the DataFrame to keep only the 15 features + target ---
# This drops columns like 'trim', 'seller', 'mileage_per_year', etc.
df = df[ALL_COLS_TO_KEEP]

In [ ]:
# --- 3. Log Transformation ---
skewed_cols = ["odometer", "car_age"]
# Use .copy() to avoid SettingWithCopyWarning
df[skewed_cols] = np.log1p(df[skewed_cols])

In [ ]:
# --- 4. Define X and y ---
# Transform the target to log-scale for training
y = np.log1p(df[TARGET_COLUMN])
X = df.drop(columns=[TARGET_COLUMN])

# Auto-detect categorical features for CatBoost
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

print(f"Final Input Features (X): {X.columns.tolist()}")
print(f"Categorical Features: {cat_features}")

Final Input Features (X): ['make', 'model', 'body', 'transmission', 'state', 'condition_category', 'odometer', 'mmr', 'car_age', 'sale_year', 'sale_month', 'seller_category']
Categorical Features: ['make', 'model', 'body', 'transmission', 'state', 'condition_category', 'seller_category']


In [ ]:
# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [2]:
# --- 1. Define the parameters and their search ranges ---
# Note: Lower learning_rate usually requires higher iterations.
# takes too long
param_dist = {
    # Basic Parameters
    'iterations': [400, 500,650, 750],  # Number of trees (iterations)
    'learning_rate': [0.01, 0.05, 0.1, 0.15], # Step size at each iteration

    # Regularization & Tree Structure
    'depth': [5, 6, 7, 8], # Max depth of the trees
    'l2_leaf_reg': [1, 3, 5, 7], # L2 regularization term
    'min_data_in_leaf': [1, 5, 10], # Minimum number of samples in a leaf

    # Overfitting prevention (Early Stopping)
    # The CatBoost model object will handle the internal early stopping

    # Other fixed parameters (passed to the model constructor)
    'loss_function': ['MAE'], # We are optimizing for MAE
    'verbose': [False], # Keep output clean during search
    'random_seed': [42],
    'cat_features': [cat_features]
}

In [1]:
# --- 2. Initialize the Base Model ---
# We use the CatBoost model without specifying iterations/LR here,
# as RandomizedSearchCV will handle those.
base_model = CatBoostRegressor(
    loss_function='MAE',
    random_seed=42,
    cat_features=cat_features,
    # Setting early_stopping_rounds helps prevent overfitting within the search
    early_stopping_rounds=50,
    verbose=0
)

# --- 3. Define Cross-Validation Strategy (K-Fold) ---
# Use 3 folds for faster initial tuning. Increase to 5 for final tuning.
cv_folds = KFold(n_splits=3, shuffle=True, random_state=42)

# --- 4. Initialize Randomized Search ---
# n_iter: How many different parameter combinations to try. Start low (20-50).
RANDOM_SEARCH_ITERATIONS = 15
print(f"Starting Randomized Search with {RANDOM_SEARCH_ITERATIONS} iterations...")

random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=RANDOM_SEARCH_ITERATIONS,
    cv=cv_folds,
    scoring='neg_mean_absolute_error', # Maximize the negative MAE (minimize MAE)
    verbose=1,
    n_jobs=-1, # Use all available cores
    random_state=42
)

# --- 5. Run the Search ---
start_time = time.time()
random_search.fit(X_train, y_train)
end_time = time.time()

print(f"\nSearch complete in {end_time - start_time:.2f} seconds.")

# --- 6. Review Results ---
best_mae = -random_search.best_score_
print(f"\nBest Cross-Validation MAE (Log-Transformed): {best_mae:.4f}")
print(f"Best Parameters Found: {random_search.best_params_}")

# --- 7. Save the Tuned Model ---
best_catboost_model = random_search.best_estimator_
dump(best_catboost_model, 'catboost_tuned_final.joblib')

print("\nBest model saved to 'catboost_tuned_final.joblib'.")

In [ ]:
# Train with Pool (efficient)
cat_features = X_train.select_dtypes(include=['object']).columns.tolist()
# Add other columns that should be treated as categorical even if their dtype is not 'object'
# e.g., 'year', 'sale_year', 'sale_month', 'is_weekend' are often treated as categorical
additional_cat_features = ['year', 'sale_year', 'sale_month', 'is_weekend']
for col in additional_cat_features:
    if col not in cat_features and col in X_train.columns:
        cat_features.append(col)

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool = Pool(X_test, y_test, cat_features=cat_features) # Create test pool for eval_set

model = CatBoostRegressor(
    iterations=700,
    depth=8,
    learning_rate=0.15,
    random_seed=42,
    early_stopping_rounds=50,
    use_best_model=True,
    verbose=100
)
model.fit(train_pool, eval_set=test_pool) # Pass eval_set to fit method

0:	learn: 0.7706914	test: 0.7678935	best: 0.7678935 (0)	total: 180ms	remaining: 2m 5s
100:	learn: 0.2045270	test: 0.2060929	best: 0.2060929 (100)	total: 13.6s	remaining: 1m 20s
200:	learn: 0.1982794	test: 0.2036203	best: 0.2036203 (200)	total: 27.1s	remaining: 1m 7s
300:	learn: 0.1945802	test: 0.2025760	best: 0.2025734 (299)	total: 41.2s	remaining: 54.6s
400:	learn: 0.1916788	test: 0.2019844	best: 0.2019778 (397)	total: 55.4s	remaining: 41.3s
500:	learn: 0.1890596	test: 0.2015077	best: 0.2015046 (498)	total: 1m 9s	remaining: 27.7s
600:	learn: 0.1868912	test: 0.2012306	best: 0.2012259 (599)	total: 1m 24s	remaining: 13.9s
699:	learn: 0.1848668	test: 0.2010445	best: 0.2010413 (696)	total: 1m 39s	remaining: 0us

bestTest = 0.2010412569
bestIteration = 696

Shrink model to first 697 iterations.


In [ ]:
y_train_pred_log = model.predict(X_train)
y_test_pred_log = model.predict(X_test)

# Actuals (already log-transformed in Cell 20)
y_train_actual_log = y.loc[X_train.index]
y_test_actual_log = y.loc[X_test.index]

# 1. Back-Transform predictions and actuals to dollar scale
y_train_actual_dollars = np.expm1(y_train_actual_log)
y_test_actual_dollars = np.expm1(y_test_actual_log)
y_train_pred_dollars = np.expm1(y_train_pred_log)
y_test_pred_dollars = np.expm1(y_test_pred_log)

# 2. Calculate metrics (MAE, RMSE) on the DOLLAR SCALE
train_mae = mean_absolute_error(y_train_actual_dollars, y_train_pred_dollars)
test_mae = mean_absolute_error(y_test_actual_dollars, y_test_pred_dollars)
train_rmse = np.sqrt(mean_squared_error(y_train_actual_dollars, y_train_pred_dollars))
test_rmse = np.sqrt(mean_squared_error(y_test_actual_dollars, y_test_pred_dollars))

print(f"Train MAE: ${train_mae:.0f} | Test MAE: ${test_mae:.0f} | Gap: {((train_mae - test_mae) / test_mae * 100):+.1f}%")
print(f"Train RMSE: ${train_rmse:.0f} | Test RMSE: ${test_rmse:.0f} | Gap: {((train_rmse - test_rmse) / test_rmse * 100):+.1f}%")

# Threshold: If test/train ratio <0.85 (test 15% worse), overfitting likely
if test_rmse / train_rmse < 0.85:
    print("⚠️ Potential Overfitting: Test underperforms train significantly.")
else:
    print("✅ Good Fit: Metrics are comparable.")

Train MAE: $980 | Test MAE: $994 | Gap: -1.5%
Train RMSE: $1824 | Test RMSE: $1820 | Gap: +0.2%
✅ Good Fit: Metrics are comparable.


In [ ]:
# Save the model
MODEL_SAVE_PATH = 'catboost_simplified.cbm'
model.save_model(MODEL_SAVE_PATH)
print(f"Successfully saved the 15-feature model to: {MODEL_SAVE_PATH}")

Successfully saved the 15-feature model to: catboost_simplified.cbm


In [ ]:
# --- Configuration ---
NEWS_FILE_PATH = "News_dataset.csv"
FAISS_INDEX_PATH = "faiss_news_index"

# --- Index Creation Function ---
def build_and_save_index():
    print(f"Starting index creation from {NEWS_FILE_PATH}...")
    try:
        df = pd.read_csv(NEWS_FILE_PATH)

        # 1. Clean and Combine Text Columns
        df['headline'] = df['headline'].fillna('')
        df['short_description'] = df['short_description'].fillna('')
        df['content'] = df['headline'].astype(str) + ". " + df['short_description'].astype(str)

        # 2. Prepare LangChain Documents
        documents = []

        # Use tqdm to show progress while processing rows
        for index, row in tqdm(df.iterrows(), total=len(df), desc="Preparing Documents"):
            if row['content'].strip() == '.':
                continue

            metadata = {
                "headline": row['headline'],
                "category": row['category'],
                "year": row['year'],
                "month": row['month'],
                "day": row['day']
            }
            documents.append(
                Document(page_content=row['content'], metadata=metadata)
            )

        # 3. Split documents
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
        docs = text_splitter.split_documents(documents)
        print(f"Total chunks created for embedding: {len(docs)}")

        # 4. Create OpenAI embeddings
        embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

        # 5. Create and save FAISS index with a progress bar
        # We wrap the list of documents with tqdm for a visual indicator

        # Note: We must convert the list comprehension to a regular list before passing it
        # to FAISS.from_documents() to allow tqdm to wrap it correctly.

        tqdm_docs = list(tqdm(docs, desc="Generating Embeddings & Building Index"))
        vector_store = FAISS.from_documents(tqdm_docs, embeddings)

        # 6. Save the index
        vector_store.save_local(FAISS_INDEX_PATH)

        print(f"\n--- SUCCESS! ---")
        print(f"FAISS index saved to: '{FAISS_INDEX_PATH}'")

    except FileNotFoundError:
        print(f"Error: The file '{NEWS_FILE_PATH}' was not found.")
    except Exception as e:
        print(f"An error occurred during indexing: {e}")

In [ ]:
build_and_save_index()

Starting index creation from News_dataset.csv...


Preparing Documents: 100%|██████████| 209527/209527 [00:12<00:00, 16929.26it/s]


Total chunks created for embedding: 209923


Generating Embeddings & Building Index: 100%|██████████| 209923/209923 [00:00<00:00, 5220727.29it/s]



--- SUCCESS! ---
FAISS index saved to: 'faiss_news_index'
